In [1]:
# Prepara reanalise ERA5 para input do SWAN
# Henrique Pereira - 14/03/2025
# ~ AtmosMarine ~

In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import cmocean

In [3]:
# windwave

ds_wind = xr.open_dataset('/mnt/c/Users/henri/OneDrive - atmosmarine.com/database/ERA5/Brasil/ERA5_Brasil_202411/data_stream-oper_stepType-instant.nc')
ds_wave = xr.open_dataset('/mnt/c/Users/henri/OneDrive - atmosmarine.com/database/ERA5/Brasil/ERA5_Brasil_202411/data_stream-wave_stepType-instant.nc')

ds_wind = ds_wind.sel(latitude=slice(-22.0, -24.0), longitude=slice(-45.0, -41.0))
ds_wave = ds_wave.sel(latitude=slice(-22.0, -24.0), longitude=slice(-45.0, -41.0))

lat_new = np.arange(-24.0, -22.0 + 0.01, 0.01)  # Inclui -22.0 no final
lon_new = np.arange(-45.0, -41.0 + 0.01, 0.01)  # Inclui -41.0 no final

ds_wind = ds_wind.interp(longitude=lon_new, latitude=lat_new)
ds_wave = ds_wave.interp(longitude=lon_new, latitude=lat_new)

# Reindexando ds1 para a grade de ds2 - tem que ser da de mais resolucao para a de menor
ds_wind = ds_wind.reindex_like(ds_wave, method='nearest') # certo - 
# ds_wave = ds_wave.reindex_like(ds_wind, method='nearest') # errado

ds_windwave = xr.merge([ds_wind, ds_wave])

ds_windwave = ds_windwave.drop_vars(['expver', 'number'])

ds_windwave = ds_windwave.rename_dims({'valid_time': 'time'})

# Para o SWAN
ds_windwave1 = ds_windwave[['u10', 'v10', 'swh', 'mwd', 'mwp']]
# Inverter a ordem da latitude e Reindexar as variáveis para manter a consistência
# ds_windwave1['latitude'] = ds_windwave1['latitude'][::-1]
ds_windwave1 = ds_windwave1.reindex(latitude=ds_windwave1['latitude'])
ds_windwave1 = ds_windwave1.rename_vars({'valid_time': 'time'})
# ds_windwave1
ds_windwave1.to_netcdf('RJ/NCfiles/windwave_rj_era5_1km_202411.nc')